In [ ]:
# CELL 1 — Verify all libraries are available
import sys
print(f"Python version: {sys.version}")

libraries = ["pandas", "numpy", "matplotlib", "seaborn", "sklearn", "joblib"]
for lib in libraries:
    try:
        __import__(lib)
        print(f"  ✅ {lib} is installed")
    except ImportError:
        print(f"  ❌ {lib} is MISSING — run: pip install {lib}")


In [ ]:
# CELL 2 — Import all required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)
import joblib

print("✅ All libraries imported successfully!")


In [ ]:
# CELL 3 — Load the LIGTAS voltage dataset
df = pd.read_csv("ligtas_voltage_dataset.csv")

print("=== DATASET LOADED ===")
print(f"Shape         : {df.shape}")
print(f"Columns       : {list(df.columns)}")
print(f"Voltage range : {df['voltage_v'].min():.2f} V  to  {df['voltage_v'].max():.2f} V")
print(f"Sensor rated  : 0 - 250 V  (ZMPT101B)")
print("\nFirst 10 rows:")
display(df.head(10))
print("\nLabel distribution:")
print(df['status'].value_counts())


In [ ]:
# CELL 4 — Design decisions and formula documentation
#
# ┌─────────────────────────────────────────────────────────────────┐
# │  SENSOR RANGE: 0 - 250V  (ZMPT101B rated range)                 │
# │  Use case: AC leakage from downed pole / transformer secondary   │
# │  spreading into flood water. Sensor reads decayed voltage at     │
# │  its placement point — not the full pole voltage.                │
# ├─────────────────────────────────────────────────────────────────┤
# │  COVERAGE AREA FORMULA — ESD Flood Water Surface Spread          │
# │  hazard_radius (m)  = V / 6.56                                   │
# │  coverage_area (m2) = pi x (V / 6.56)^2                          │
# │  Basis: Rifkin & Shafer (2008) Electric Shock Drowning Study     │
# │  US Coast Guard — lethal gradient = 6.56 V/m (2 V/ft)           │
# │  Ref  : https://en.wikipedia.org/wiki/Electric_shock_drowning    │
# ├─────────────────────────────────────────────────────────────────┤
# │  CLASSIFICATION THRESHOLDS                                       │
# │    0  - 25V  →  Safe      (below ESD lethal threshold)           │
# │    25.01 - 29.99V → Warning   (approaching danger)               │
# │    30 - 250V →  Dangerous (exceeds safe exposure limit)          │
# ├─────────────────────────────────────────────────────────────────┤
# │  FEATURES (6) — spike_flag and voltage_delta removed             │
# │                                                                  │
# │  REMOVED: voltage_delta  — sensor reads every <1s, delta ≈ 0    │
# │  REMOVED: spike_flag     — same alert regardless of spike/steady │
# │                                                                  │
# │  KEPT:                                                           │
# │    voltage_v           — raw sensor reading                      │
# │    voltage_squared     — V^2, energy scaling                     │
# │    coverage_area_m2    — ESD hazard zone formula                 │
# │    voltage_class       — zone category (0-4)                     │
# │    danger_score        — combined risk index                     │
# │    sensor_noise_level  — ZMPT101B ~2% noise                      │
# └─────────────────────────────────────────────────────────────────┘

print("=== DESIGN VERIFICATION ===")
import numpy as np
ESD_GRADIENT = 6.56
print(f"\nVoltage range  : {df['voltage_v'].min():.2f} to {df['voltage_v'].max():.2f} V")
print(f"Features       : {['voltage_v','voltage_squared','coverage_area_m2','voltage_class','danger_score','sensor_noise_level']}")
print(f"\nCoverage area samples:")
for v in [1, 5, 10, 25, 28, 30, 100, 220, 250]:
    r = v / ESD_GRADIENT
    a = np.pi * r * r
    lbl = 'Safe' if v <= 25.0 else ('Warning' if v < 30.0 else 'Dangerous')
    print(f"  V={v:>4}V  radius={r:.2f}m  area={a:>8.2f} m2  [{lbl}]")
print(f"\nLabel distribution:")
print(df['status'].value_counts())


In [ ]:
# CELL 5 — Data exploration
print("=== DATASET SUMMARY ===")
display(df.describe())

print("\n=== CLASS DISTRIBUTION ===")
print(df['status'].value_counts())
print("\n=== RISK FLAG DISTRIBUTION ===")
print(df['risk_flag'].value_counts())


In [ ]:
# CELL 6 — Data visualization
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("LIGTAS Dataset — 0-250V, ESD Formula, 6 Features", fontsize=14, fontweight="bold")

ESD_GRADIENT = 6.56

# Plot 1: Voltage distribution
axes[0,0].hist(df["voltage_v"], bins=60, color="steelblue", edgecolor="black", alpha=0.8)
axes[0,0].axvline(x=25, color="orange", linestyle="--", linewidth=1.5, label="25V  Warning")
axes[0,0].axvline(x=30, color="red",    linestyle="--", linewidth=2,   label="30V  Danger")
axes[0,0].set_title("Voltage Distribution (0-250V)")
axes[0,0].set_xlabel("Voltage (V)")
axes[0,0].set_ylabel("Count")
axes[0,0].legend()

# Plot 2: Class distribution
colors = ["#2ecc71", "#f39c12", "#e74c3c"]
counts = df["status"].value_counts().reindex(["Safe","Warning","Dangerous"])
axes[0,1].bar(counts.index, counts.values, color=colors, edgecolor="black")
axes[0,1].set_title("Class Distribution")
axes[0,1].set_xlabel("Status")
axes[0,1].set_ylabel("Number of Samples")
for i, v in enumerate(counts.values):
    axes[0,1].text(i, v + 20, str(v), ha="center", fontweight="bold")

# Plot 3: ESD coverage area curve
base   = df.drop_duplicates("voltage_v").sort_values("voltage_v")
sample = base[base["voltage_v"] <= 250]
axes[0,2].plot(sample["voltage_v"], sample["coverage_area_m2"], color="steelblue", linewidth=2)
axes[0,2].axvline(x=25, color="orange", linestyle="--", linewidth=1.5, label="25V  Warning")
axes[0,2].axvline(x=30, color="red",    linestyle="--", linewidth=2,   label="30V  Danger")
axes[0,2].fill_between(sample["voltage_v"], sample["coverage_area_m2"],
                        where=(sample["voltage_v"] <= 25),
                        color="green",  alpha=0.2, label="Safe")
axes[0,2].fill_between(sample["voltage_v"], sample["coverage_area_m2"],
                        where=((sample["voltage_v"] > 25) & (sample["voltage_v"] < 30)),
                        color="orange", alpha=0.2, label="Warning")
axes[0,2].fill_between(sample["voltage_v"], sample["coverage_area_m2"],
                        where=(sample["voltage_v"] >= 30),
                        color="red",    alpha=0.2, label="Dangerous (30-250V)")
axes[0,2].set_title("A = pi x (V/6.56)^2  ESD Flood Water Formula")
axes[0,2].set_xlabel("Voltage V")
axes[0,2].set_ylabel("Hazard Surface Area (m2)")
axes[0,2].legend(fontsize=8)
axes[0,2].grid(True, linestyle="--", alpha=0.5)

# Plot 4: Danger score vs voltage
axes[1,0].scatter(df["voltage_v"], df["danger_score"],
                   c=df["label"].map({0:"green",1:"orange",2:"red"}),
                   alpha=0.3, s=8)
axes[1,0].axvline(x=25, color="orange", linestyle="--", linewidth=1.5)
axes[1,0].axvline(x=30, color="red",    linestyle="--", linewidth=2)
axes[1,0].set_title("Danger Score vs Voltage")
axes[1,0].set_xlabel("Voltage (V)")
axes[1,0].set_ylabel("Danger Score")
axes[1,0].grid(True, linestyle="--", alpha=0.4)

# Plot 5: Sensor noise level
axes[1,1].hist(df["sensor_noise_level"], bins=50, color="#e67e22", edgecolor="black", alpha=0.8)
axes[1,1].set_title("Sensor Noise Level (ZMPT101B ~2%)")
axes[1,1].set_xlabel("Noise Level (V)")
axes[1,1].set_ylabel("Count")

# Plot 6: Borderline zone 23-35V
border = df[(df["voltage_v"] >= 23) & (df["voltage_v"] <= 35)]
colors_map = {0: "green", 1: "orange", 2: "red"}
axes[1,2].scatter(border["voltage_v"], border["coverage_area_m2"],
                   c=border["label"].map(colors_map), alpha=0.35, s=12)
axes[1,2].axvline(x=25, color="orange", linestyle="--", linewidth=1.5, label="25V Warning")
axes[1,2].axvline(x=30, color="black",  linestyle="--", linewidth=2,   label="30V Danger")
axes[1,2].set_title(f"Borderline Zone 23-35V ({len(border)} rows)")
axes[1,2].set_xlabel("Voltage V")
axes[1,2].set_ylabel("Coverage Area (m2)")
axes[1,2].legend()
axes[1,2].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig("ligtas_overview.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Chart saved as ligtas_overview.png")


In [ ]:
# CELL 7 — Prepare features (X) and labels (y)
#
# 6 features:
#
#  Feature              Reason
#  ───────────────────  ──────────────────────────────────────────────────
#  voltage_v            Raw sensor reading (ZMPT101B, 0-250V)
#  voltage_squared      V^2 — energy scales with square of voltage
#  coverage_area_m2     pi x (V/6.56)^2 — ESD flood water surface spread
#  voltage_class        0-4 intensity zone category
#  danger_score         V / (1 + log(area+1)) — combined risk index
#  sensor_noise_level   Estimated ZMPT101B noise (2% of reading)
#
# Removed:
#  voltage_delta  — sensor reads every <1s, delta always near zero
#  spike_flag     — same alert regardless of spike vs steady reading
#
# Label rules:
#   0 = Safe      (0  - 25V)
#   1 = Warning   (25.01 - 29.99V)
#   2 = Dangerous (30 - 250V)

# Re-classify labels
def reclassify(v):
    if v <= 25.0:    return 0
    elif v < 30.0:   return 1
    else:            return 2

df['label']     = df['voltage_v'].apply(reclassify)
df['status']    = df['label'].map({0: 'Safe', 1: 'Warning', 2: 'Dangerous'})
df['risk_flag'] = df['label'].map({0: 'SAFE', 1: 'WARNING', 2: 'DANGER'})

ESD_GRADIENT = 6.56
df['coverage_area_m2'] = np.pi * (df['voltage_v'] / ESD_GRADIENT) ** 2
df['danger_score']     = df['voltage_v'] / (1.0 + np.log(df['coverage_area_m2'] + 1.0))

X = df[[
    "voltage_v",
    "voltage_squared",
    "coverage_area_m2",
    "voltage_class",
    "danger_score",
    "sensor_noise_level"
]]
y = df["label"]

print("=== FEATURES (X) — 6 features ===")
display(X.head(10))
print("\n=== LABELS (y) ===")
print(f"  0 = Safe      : {(y==0).sum():,}")
print(f"  1 = Warning   : {(y==1).sum():,}")
print(f"  2 = Dangerous : {(y==2).sum():,}")
print(f"\n  Total         : {len(y):,}")


In [ ]:
# CELL 8 — Train/Test split (80/20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("=== TRAIN / TEST SPLIT ===")
print(f"  Training: {len(X_train):,}  (80%)")
print(f"  Testing : {len(X_test):,}  (20%)")
print("\nTrain distribution:")
for lbl, name in [(0,"Safe"),(1,"Warning"),(2,"Dangerous")]:
    print(f"  {name:<10}: {(y_train==lbl).sum():,}")
print("\nTest distribution:")
for lbl, name in [(0,"Safe"),(1,"Warning"),(2,"Dangerous")]:
    print(f"  {name:<10}: {(y_test==lbl).sum():,}")


In [ ]:
# CELL 9 — Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("✅ Features scaled (6 features)")
print("\nScaler means:")
for col, mean in zip(X.columns, scaler.mean_):
    print(f"  {col:<22}: {mean:.4f}")


In [ ]:
# CELL 10 — Train Random Forest model
print("Training Random Forest (100 trees, 6 features)...")

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)
rf_model.fit(X_train_scaled, y_train)

print("✅ Random Forest training complete!")
print(f"   Trees    : {rf_model.n_estimators}")
print(f"   Features : {rf_model.n_features_in_}")
print(f"   Classes  : {list(rf_model.classes_)}")

importances = rf_model.feature_importances_
print("\n=== FEATURE IMPORTANCE ===")
for name, imp in sorted(zip(X.columns, importances), key=lambda x: -x[1]):
    bar = "█" * int(imp * 60)
    print(f"  {name:<22}: {imp:.4f}  {bar}")


In [ ]:
# CELL 11 — Evaluate Random Forest model
rf_pred = rf_model.predict(X_test_scaled)

rf_acc  = accuracy_score(y_test,  rf_pred)
rf_prec = precision_score(y_test, rf_pred, average="weighted", zero_division=0)
rf_rec  = recall_score(y_test,    rf_pred, average="weighted", zero_division=0)
rf_f1   = f1_score(y_test,        rf_pred, average="weighted", zero_division=0)

print("=" * 55)
print("  MODEL: Random Forest — 6 features, ESD formula")
print("=" * 55)
print(f"  Accuracy   : {rf_acc:.4f}  ({rf_acc*100:.2f}%)")
print(f"  Precision  : {rf_prec:.4f}")
print(f"  Recall     : {rf_rec:.4f}")
print(f"  F1-Score   : {rf_f1:.4f}")
print(f"\nDetailed Report:")
print(classification_report(y_test, rf_pred,
      target_names=["Safe (0)","Warning (1)","Dangerous (2)"]))


In [ ]:
# CELL 12 — Model Metrics Chart
metrics   = ["Accuracy", "Precision", "Recall", "F1-Score"]
rf_scores = [rf_acc, rf_prec, rf_rec, rf_f1]
bar_colors = ["#3b82f6", "#22c55e", "#f59e0b", "#ef4444"]

fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor("#0d1117")
ax.set_facecolor("#161b22")

bars = ax.bar(metrics, rf_scores, color=bar_colors, edgecolor="white",
              linewidth=1.2, width=0.5, zorder=3)

for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.005,
            f"{h:.4f}", ha="center", va="bottom",
            fontsize=11, color="white", fontweight="bold")

ax.set_ylim(0, 1.15)
ax.set_xticklabels(metrics, color="#e5e7eb", fontsize=12, fontweight="bold")
ax.set_ylabel("Score", color="#9ca3af", fontsize=11)
ax.set_title("LIGTAS — Random Forest Performance (All Gaps Fixed)",
             color="#f9fafb", fontsize=13, fontweight="bold", pad=14)
ax.tick_params(colors="#6b7280")
for spine in ax.spines.values(): spine.set_color("#30363d")
ax.set_yticks([0.0,0.2,0.4,0.6,0.8,1.0])
ax.set_yticklabels(["0.0","0.2","0.4","0.6","0.8","1.0"], color="#6b7280")
ax.axhline(y=1.0, color="#374151", linestyle=":", linewidth=1.2, zorder=2)
ax.grid(axis="y", color="#21262d", linewidth=0.8, linestyle="--", zorder=1)

plt.tight_layout()
plt.savefig("model_metrics.png", dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show()
print("✅ Metrics chart saved as model_metrics.png")


In [ ]:
# CELL 13 — Confusion Matrix
fig, ax = plt.subplots(figsize=(7, 5))
fig.suptitle("LIGTAS — Confusion Matrix (Random Forest, 6 Features)",
             fontsize=12, fontweight="bold")

cm = confusion_matrix(y_test, rf_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
            xticklabels=["Safe","Warning","Dangerous"],
            yticklabels=["Safe","Warning","Dangerous"],
            linewidths=0.5)
ax.set_xlabel("Predicted Label", fontsize=10)
ax.set_ylabel("Actual Label",    fontsize=10)

print("Per-class summary:")
for i, cls in enumerate(["Safe","Warning","Dangerous"]):
    tp = cm[i,i]
    fp = cm[:,i].sum() - tp
    fn = cm[i,:].sum() - tp
    tn = cm.sum() - tp - fp - fn
    print(f"  {cls:<10}: TP={tp}  TN={tn}  FP={fp}  FN={fn}")

plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Confusion matrix saved as confusion_matrix.png")


In [ ]:
# CELL 14 — Coverage Area Curve (ESD Flood Water Formula, 0-250V)
# Formula: A = pi x (V / 6.56)^2
# Basis  : Rifkin & Shafer (2008) Electric Shock Drowning Study
#          US Coast Guard — lethal gradient = 6.56 V/m (2 V/ft)
# Ref    : https://en.wikipedia.org/wiki/Electric_shock_drowning
# Note   : 250V is the ZMPT101B sensor max and is within the Dangerous zone.
#          No separate 250V marker — everything 30V+ is red.

ESD_GRADIENT = 6.56
base = df.drop_duplicates("voltage_v").sort_values("voltage_v")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    "LIGTAS — Hazard Coverage Area  A = pi x (V/6.56)^2  (0-250V, ESD Formula)",
    fontsize=13, fontweight="bold"
)

for ax, (title, data) in zip(axes, [
    ("Full Range (0-250V)", base),
    ("Zoomed (0-100V)",     base[base["voltage_v"] <= 100])
]):
    ax.plot(data["voltage_v"], data["coverage_area_m2"], color="steelblue", linewidth=2)
    ax.axvline(x=25, color="orange", linestyle="--", linewidth=1.5, label="25V  Warning")
    ax.axvline(x=30, color="red",    linestyle="--", linewidth=2,   label="30V  Danger")
    ax.fill_between(data["voltage_v"], data["coverage_area_m2"],
                     where=(data["voltage_v"] <= 25),
                     color="green",  alpha=0.25, label="Safe")
    ax.fill_between(data["voltage_v"], data["coverage_area_m2"],
                     where=((data["voltage_v"] > 25) & (data["voltage_v"] < 30)),
                     color="orange", alpha=0.2,  label="Warning")
    ax.fill_between(data["voltage_v"], data["coverage_area_m2"],
                     where=(data["voltage_v"] >= 30),
                     color="red",    alpha=0.2,  label="Dangerous (30-250V)")
    ax.set_title(title)
    ax.set_xlabel("Voltage V (sensor reading in flood water)")
    ax.set_ylabel("Hazard Surface Area (m2)")
    ax.legend(fontsize=8)
    ax.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig("coverage_area_curve.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Coverage curve saved as coverage_area_curve.png")
print()
print("Sample values (ZMPT101B 0-250V range):")
for v in [1, 5, 10, 25, 28, 30, 100, 220, 250]:
    r = v / ESD_GRADIENT
    a = np.pi * r * r
    lbl = "Safe" if v <= 25 else ("Warning" if v <= 29 else "Dangerous")
    print(f"  V={v:>4}V  radius={r:.2f}m  area={a:>8.2f} m2  [{lbl}]")


In [ ]:
# CELL 15 — Save trained model and scaler
joblib.dump(rf_model, "ligtas_rf_model.pkl")
joblib.dump(scaler,   "ligtas_scaler.pkl")
print("✅ Saved:")
print("   ligtas_rf_model.pkl — Random Forest (6 features)")
print("   ligtas_scaler.pkl   — StandardScaler (6 features)")


In [ ]:
# CELL 16 — Real-time prediction function (6 features, 0-250V)
ESD_GRADIENT = 6.56

def ligtas_predict(voltage_value):
    voltage_value = min(max(voltage_value, 0.0), 250.0)  # clamp to sensor range

    v_sq         = voltage_value ** 2
    radius        = voltage_value / ESD_GRADIENT
    area          = np.pi * radius * radius
    vc            = (0 if voltage_value < 5 else
                     1 if voltage_value < 26 else
                     2 if voltage_value < 30 else
                     3 if voltage_value < 100 else 4)
    danger_score  = voltage_value / (1.0 + np.log(area + 1.0))
    noise_level   = voltage_value * 0.02

    input_scaled = scaler.transform([[
        voltage_value, v_sq, area, vc, danger_score, noise_level
    ]])
    pred = rf_model.predict(input_scaled)[0]

    icons = {0: "🟢 SAFE", 1: "🟡 WARNING", 2: "🔴 DANGEROUS"}
    print(f"  ┌──────────────────────────────────────────────────")
    print(f"  │  Voltage          : {voltage_value} V")
    print(f"  │  Hazard Radius    : {radius:.2f} m")
    print(f"  │  Coverage Area    : {area:.2f} m2")
    print(f"  │  Danger Score     : {danger_score:.4f}")
    print(f"  │  Classification   : {icons[pred]}")
    print(f"  └──────────────────────────────────────────────────")

print("=" * 55)
print("  LIGTAS — Prediction Test (6 features, 0-250V)")
print("=" * 55)
test_cases = [
    (0.0,   "no leakage"),
    (5.0,   "weak leakage"),
    (10.0,  "low leakage"),
    (25.0,  "at safe boundary"),
    (26.0,  "just entered warning zone"),
    (29.9,  "just below danger threshold"),
    (30.0,  "just crossed danger threshold"),
    (100.0, "high voltage leakage"),
    (220.0, "mains secondary side leak"),
    (250.0, "sensor maximum reading"),
]
for v, desc in test_cases:
    print(f"\n  [{desc}]")
    ligtas_predict(v)


In [ ]:
# CELL 17 — Load saved model without retraining
loaded_model  = joblib.load("ligtas_rf_model.pkl")
loaded_scaler = joblib.load("ligtas_scaler.pkl")
print("✅ Model and scaler loaded!")
print(f"   Features : 6  (voltage_delta and spike_flag removed)")
print(f"   Range    : 0-250V  (ZMPT101B)")

ESD_GRADIENT = 6.56

test_v = 220.0
v_sq   = test_v ** 2
radius = test_v / ESD_GRADIENT
area   = np.pi * radius * radius
vc     = 4
ds     = test_v / (1.0 + np.log(area + 1.0))
noise  = test_v * 0.02

result = loaded_model.predict(
    loaded_scaler.transform([[test_v, v_sq, area, vc, ds, noise]])
)[0]
print(f"\n220V mains leak test:")
print(f"  Hazard radius : {radius:.2f} m")
print(f"  Coverage area : {area:.2f} m2")
print(f"  Result        : {['🟢 SAFE','🟡 WARNING','🔴 DANGEROUS'][result]}")


In [ ]:
# CELL 18 — Export to Arduino C++ (ligtas_model.h)
import joblib, numpy as np
from micromlgen import port

rf_model = joblib.load("ligtas_rf_model.pkl")
scaler   = joblib.load("ligtas_scaler.pkl")

with open("ligtas_model.h", "w") as f:
    f.write(port(rf_model))
print("✅ ligtas_model.h created!")

means  = ", ".join([f"{m:.6f}f" for m in scaler.mean_])
scales = ", ".join([f"{s:.6f}f" for s in scaler.scale_])
print("\n=== COPY THESE INTO ligtas_ml.h ===")
print(f"const float SCALER_MEAN[6]  = {{{means}}};")
print(f"const float SCALER_SCALE[6] = {{{scales}}};")
print("\n// Feature order [0-5]:")
print("// [0] voltage_v          — raw sensor reading")
print("// [1] voltage_squared    — V^2")
print("// [2] coverage_area_m2   — pi x (V/6.56)^2")
print("// [3] voltage_class      — 0-4 zone")
print("// [4] danger_score       — V / (1 + log(area+1))")
print("// [5] sensor_noise_level — V x 0.02")
